In [27]:
# Import packages and load the cleaned TCGA-BRCA clinical table

import pandas as pd
import numpy as np
import requests
import json

clinical_df = pd.read_csv("../data/processed/tcga_brca_clinical.csv")

print("Clinical rows:", len(clinical_df))

clinical_df.head()

Clinical rows: 1098


,case_id,submitter_id,sex_at_birth,vital_status,age_at_diagnosis_days,ajcc_pathologic_stage,age_at_diagnosis_years
0,57a1604c-60b7-4b30-a75e-f70939532c5c,TCGA-BH-A0B2,NaN,NaN,NaN,NaN,NaN
1,2779fa01-ac93-4e80-a997-3385f72172c3,TCGA-A8-A08S,female,Alive,25994.0,Stage IIA,71.2
2,80f3f48a-cc21-415c-b181-c77e7ba1c563,TCGA-A8-A09C,female,Alive,25355.0,Stage X,69.4
3,e663954c-05a8-479a-a623-400c1eb5efc6,TCGA-BH-A1F5,female,Dead,22842.0,Stage IIA,62.5
4,14b95463-2108-4921-afc2-e29eef52b18f,TCGA-A2-A0YT,female,Dead,20482.0,Stage IIIB,56.1


In [28]:
# Find TCGA-BRCA somatic mutation files in the GDC

files_url = "https://api.gdc.cancer.gov/files"

mutation_filters = {
    "op": "and",
    "content": [
        {
            "op": "=",
            "content": {
                "field": "cases.project.project_id",
                "value": "TCGA-BRCA"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "data_category",
                "value": "Simple Nucleotide Variation"
            }
        }
    ]
}

mutation_params = {
    "filters": json.dumps(mutation_filters),
    "fields": "file_id,file_name,data_type,data_format,experimental_strategy",
    "format": "JSON",
    "size": 1000
}

mutation_files_response = requests.get(
    files_url,
    params=mutation_params
)

print("Status code:", mutation_files_response.status_code)

Status code: 200


In [29]:
# Inspect the TCGA-BRCA somatic mutation files returned by the GDC

mutation_files_json = mutation_files_response.json()

mutation_files = mutation_files_json["data"]["hits"]

print("Number of mutation files found:", len(mutation_files))

mutation_files[:5]

Number of mutation files found: 1000


[{'id': '6f1957a7-d481-4419-a5f4-e51d05cbd6d9',
  'data_format': 'MAF',
  'file_name': '4840f7f8-32e3-407a-9919-1e1f8f656533.wxs.Pindel.aliquot.maf.gz',
  'file_id': '6f1957a7-d481-4419-a5f4-e51d05cbd6d9',
  'data_type': 'Annotated Somatic Mutation',
  'experimental_strategy': 'WXS'},
 {'id': '9408f1ee-f2cf-4278-9e7a-b39edfc83aee',
  'data_format': 'VCF',
  'file_name': '919d0196-6b75-4dfe-84b5-139b78cdd458.wxs.muse.raw_somatic_mutation.vcf.gz',
  'file_id': '9408f1ee-f2cf-4278-9e7a-b39edfc83aee',
  'data_type': 'Raw Simple Somatic Mutation',
  'experimental_strategy': 'WXS'},
 {'id': '04e31c6d-2e6c-4d25-9a4a-ff7e6866b084',
  'data_format': 'VCF',
  'file_name': 'TCGA_BRCA.4238f844-0759-4434-a0a0-fa7f8221801a.wgs.GATK4_MuTect2_Pair.somatic_annotation.vcf.gz',
  'file_id': '04e31c6d-2e6c-4d25-9a4a-ff7e6866b084',
  'data_type': 'Annotated Somatic Mutation',
  'experimental_strategy': 'WGS'},
 {'id': '081c1afc-ea50-4773-9402-dfc686603a16',
  'data_format': 'MAF',
  'file_name': 'e35fa380-

In [30]:
# Find masked whole-exome somatic mutation MAF files for TCGA-BRCA

maf_filters = {
    "op": "and",
    "content": [
        {
            "op": "=",
            "content": {
                "field": "cases.project.project_id",
                "value": "TCGA-BRCA"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "data_type",
                "value": "Masked Somatic Mutation"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "data_format",
                "value": "MAF"
            }
        },
        {
            "op": "=",
            "content": {
                "field": "experimental_strategy",
                "value": "WXS"
            }
        }
    ]
}

maf_params = {
    "filters": json.dumps(maf_filters),
    "fields": "file_id,file_name,data_type,data_format,experimental_strategy",
    "format": "JSON",
    "size": 1
}

maf_response = requests.get(
    files_url,
    params=maf_params
)

maf_json = maf_response.json()

print("Status code:", maf_response.status_code)
print("Total matching MAF files:", maf_json["data"]["pagination"]["total"])

Status code: 200
Total matching MAF files: 992


In [31]:
# Retrieve metadata for all TCGA-BRCA masked somatic mutation MAF files

maf_params = {
    "filters": json.dumps(maf_filters),
    "fields": "file_id,file_name,data_type,data_format,experimental_strategy",
    "format": "JSON",
    "size": 992
}

maf_response = requests.get(
    files_url,
    params=maf_params
)

maf_files = maf_response.json()["data"]["hits"]

print("Number of MAF files retrieved:", len(maf_files))

Number of MAF files retrieved: 992


In [32]:
# Retrieve case identifiers linked to each TCGA-BRCA mutation MAF file

maf_case_params = {
    "filters": json.dumps(maf_filters),
    "fields": "file_id,file_name,cases.case_id,cases.submitter_id",
    "format": "JSON",
    "size": 992
}

maf_case_response = requests.get(
    files_url,
    params=maf_case_params
)

print("Status code:", maf_case_response.status_code)

Status code: 200


In [33]:
# Inspect one mutation MAF file together with its linked TCGA case

maf_case_data = maf_case_response.json()["data"]["hits"]

print("Number of mutation-file records:", len(maf_case_data))
print()
print(maf_case_data[0])

Number of mutation-file records: 992

{'id': '081c1afc-ea50-4773-9402-dfc686603a16', 'cases': [{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea', 'submitter_id': 'TCGA-A8-A09E'}], 'file_name': 'e35fa380-07ac-45e1-9adc-4351675c6ccf.wxs.aliquot_ensemble_masked.maf.gz', 'file_id': '081c1afc-ea50-4773-9402-dfc686603a16'}


In [34]:
# Check how many TCGA cases are linked to each mutation MAF file

case_counts_per_file = [
    len(record.get("cases", []))
    for record in maf_case_data
]

pd.Series(case_counts_per_file).value_counts().sort_index()

1    992
Name: count, dtype: int64

In [35]:
# Build a clean mapping table linking each mutation MAF file to its TCGA case

maf_mapping_rows = []

for record in maf_case_data:
    case_info = record["cases"][0]

    maf_mapping_rows.append({
        "file_id": record["file_id"],
        "file_name": record["file_name"],
        "case_id": case_info["case_id"],
        "submitter_id": case_info["submitter_id"]
    })

maf_mapping_df = pd.DataFrame(maf_mapping_rows)

print("Rows:", len(maf_mapping_df))

maf_mapping_df.head()

Rows: 992


,file_id,file_name,case_id,submitter_id
0,081c1afc-ea50-4773-9402-dfc686603a16,e35fa380-07ac-45e1-9adc-4351675c6ccf.wxs.aliquot_ensemble_masked.maf.gz,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,TCGA-A8-A09E
1,d6709b52-b2fb-4b89-89d4-a426de3f21f0,ac3dbb7f-2ce9-43b7-9f45-1c1521e6bbf5.wxs.aliquot_ensemble_masked.maf.gz,09765b0a-94f6-47d2-af56-93368084ac3a,TCGA-A7-A0CD
2,290c5b56-3e55-4bbc-9bcd-2ef0b678490d,a329335e-2962-485d-af32-1f6adf48cf7f.wxs.aliquot_ensemble_masked.maf.gz,0a017f15-1c6b-45e7-8d55-e0a71df1b2e8,TCGA-A8-A08F
3,a89b041d-e6cd-4298-93b9-13c62a5f2f01,fe1694df-4aca-4a23-bbf5-c0c1d24d4c2b.wxs.aliquot_ensemble_masked.maf.gz,2c04a2f5-321e-4dea-8e00-268325da65cb,TCGA-A8-A06T
4,7226e23d-6c48-4a24-9c32-f61f3bde7e06,c107d4de-6001-4027-a7f1-7c03d484ddaf.wxs.aliquot_ensemble_masked.maf.gz,791c5768-f0f5-4ab6-86eb-998e5c4b49e3,TCGA-E2-A1L7


In [36]:
# Check whether each mutation MAF file corresponds to a unique TCGA-BRCA patient

print("Mutation files:", len(maf_mapping_df))
print("Unique case_id values:", maf_mapping_df["case_id"].nunique())
print("Unique submitter_id values:", maf_mapping_df["submitter_id"].nunique())

Mutation files: 992
Unique case_id values: 969
Unique submitter_id values: 969


In [37]:
# Identify TCGA-BRCA patients that have more than one mutation MAF file

files_per_patient = (
    maf_mapping_df["submitter_id"]
    .value_counts()
)

files_per_patient.value_counts().sort_index()

count
1    951
2     13
3      5
Name: count, dtype: int64

In [38]:
# Inspect TCGA-BRCA patients that have multiple mutation MAF files

patients_with_multiple_files = files_per_patient[
    files_per_patient > 1
].index

multiple_maf_df = (
    maf_mapping_df[
        maf_mapping_df["submitter_id"].isin(patients_with_multiple_files)
    ]
    .sort_values(["submitter_id", "file_name"])
)

print("Patients with multiple MAF files:", len(patients_with_multiple_files))
print("MAF files belonging to these patients:", len(multiple_maf_df))

multiple_maf_df

Patients with multiple MAF files: 18
MAF files belonging to these patients: 41


,file_id,file_name,case_id,submitter_id
911,2c9b0f52-0abf-4094-96cb-8511d091ab66,9f77e806-537f-40e9-b33b-b28af74a7a7b.wxs.aliquot_ensemble_masked.maf.gz,f130f376-5801-40f9-975d-a7e2f7b5670d,TCGA-A7-A0DB
930,3e7b34fb-a86b-40dc-b349-650301e66e1c,c6810404-b1aa-4669-b91a-d720e93635ec.wxs.aliquot_ensemble_masked.maf.gz,f130f376-5801-40f9-975d-a7e2f7b5670d,TCGA-A7-A0DB
448,a8976db3-5077-4b8f-9d6c-41a562ae1b4c,d577f725-e910-4a77-89e9-2e982d5493e5.wxs.aliquot_ensemble_masked.maf.gz,f130f376-5801-40f9-975d-a7e2f7b5670d,TCGA-A7-A0DB
968,1425f01f-b090-48c7-897c-6807939dcba2,62a96ff1-e401-48ee-9f27-b308d672b35c.wxs.aliquot_ensemble_masked.maf.gz,8785012f-f73e-4d68-87cf-1d804af32782,TCGA-A7-A13D
974,8f94cc7d-9989-4445-b0b2-90c6d6cb61fa,6a0c8b0a-52d1-4cbb-89cb-06808ee8e7e9.wxs.aliquot_ensemble_masked.maf.gz,8785012f-f73e-4d68-87cf-1d804af32782,TCGA-A7-A13D
980,9a32d563-cd69-4ea5-9b50-c5556d6fda42,764de0ed-ea23-427a-9005-5c5fab299ecd.wxs.aliquot_ensemble_masked.maf.gz,8785012f-f73e-4d68-87cf-1d804af32782,TCGA-A7-A13D
188,26fcd5cb-2178-4884-99e1-630e5f6bf722,09ebc616-2bb3-4779-b7bb-6a4d37cf3d39.wxs.aliquot_ensemble_masked.maf.gz,8c7e74e0-71ef-49b8-9217-94b8ef740ef9,TCGA-A7-A13E
224,d8ff1cbb-c551-4585-bdd1-42721e326933,26d5a3d5-2698-4791-8a89-e05ffde7cdca.wxs.aliquot_ensemble_masked.maf.gz,8c7e74e0-71ef-49b8-9217-94b8ef740ef9,TCGA-A7-A13E
229,9d6db1ea-008d-4f37-b01f-6fb7999a05ba,af7cded7-8cba-42e8-9204-3d50ba53fbe4.wxs.aliquot_ensemble_masked.maf.gz,8c7e74e0-71ef-49b8-9217-94b8ef740ef9,TCGA-A7-A13E
743,4c9931f6-8790-497c-8462-9f3ba7bcb9a2,4c6fadb5-27cf-4d3e-9a84-f4744a64f9e5.wxs.aliquot_ensemble_masked.maf.gz,6cd9baf5-bbe0-4c1e-a87f-c53b3af22890,TCGA-A7-A13G


In [39]:
# Retrieve sample information for TCGA-BRCA mutation MAF files

maf_sample_params = {
    "filters": json.dumps(maf_filters),
    "fields": (
        "file_id,file_name,"
        "cases.case_id,cases.submitter_id,"
        "cases.samples.submitter_id,"
        "cases.samples.sample_type"
    ),
    "format": "JSON",
    "size": 992
}

maf_sample_response = requests.get(
    files_url,
    params=maf_sample_params
)

print("Status code:", maf_sample_response.status_code)
print("Warnings:", maf_sample_response.json().get("warnings"))

Status code: 200
Warnings: {}


In [40]:
# Inspect one TCGA-BRCA mutation MAF file with linked sample information

maf_sample_data = maf_sample_response.json()["data"]["hits"]

print(maf_sample_data[0])

{'id': '081c1afc-ea50-4773-9402-dfc686603a16', 'cases': [{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea', 'submitter_id': 'TCGA-A8-A09E', 'samples': [{'submitter_id': 'TCGA-A8-A09E-10A', 'sample_type': 'Blood Derived Normal'}, {'submitter_id': 'TCGA-A8-A09E-01A', 'sample_type': 'Primary Tumor'}]}], 'file_name': 'e35fa380-07ac-45e1-9adc-4351675c6ccf.wxs.aliquot_ensemble_masked.maf.gz', 'file_id': '081c1afc-ea50-4773-9402-dfc686603a16'}


In [41]:
# Retrieve the exact biospecimen entities associated with each mutation MAF file

maf_entity_params = {
    "filters": json.dumps(maf_filters),
    "fields": (
        "file_id,file_name,"
        "cases.case_id,cases.submitter_id,"
        "associated_entities.entity_id,"
        "associated_entities.entity_submitter_id,"
        "associated_entities.entity_type"
    ),
    "format": "JSON",
    "size": 992
}

maf_entity_response = requests.get(
    files_url,
    params=maf_entity_params
)

print("Status code:", maf_entity_response.status_code)
print("Warnings:", maf_entity_response.json().get("warnings"))

Status code: 200
Warnings: {}


In [42]:
# Inspect one mutation MAF file with its exact associated biospecimen entity

maf_entity_data = maf_entity_response.json()["data"]["hits"]

print(maf_entity_data[0])

{'id': '081c1afc-ea50-4773-9402-dfc686603a16', 'cases': [{'case_id': 'dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea', 'submitter_id': 'TCGA-A8-A09E'}], 'associated_entities': [{'entity_submitter_id': 'TCGA-A8-A09E-10A-01W-A021-09', 'entity_type': 'aliquot', 'entity_id': '950408ef-e597-4d7c-852d-865fbc66f75d'}, {'entity_submitter_id': 'TCGA-A8-A09E-01A-11W-A019-09', 'entity_type': 'aliquot', 'entity_id': 'd6465963-5ea6-44a5-96b0-dff0b0fae4c4'}], 'file_name': 'e35fa380-07ac-45e1-9adc-4351675c6ccf.wxs.aliquot_ensemble_masked.maf.gz', 'file_id': '081c1afc-ea50-4773-9402-dfc686603a16'}


In [43]:
# Build a table linking each mutation MAF file to its associated aliquots

maf_entity_rows = []

for record in maf_entity_data:
    case_info = record["cases"][0]

    for entity in record.get("associated_entities", []):
        maf_entity_rows.append({
            "file_id": record["file_id"],
            "file_name": record["file_name"],
            "case_id": case_info["case_id"],
            "submitter_id": case_info["submitter_id"],
            "entity_submitter_id": entity.get("entity_submitter_id"),
            "entity_type": entity.get("entity_type")
        })

maf_entity_df = pd.DataFrame(maf_entity_rows)

print("Rows:", len(maf_entity_df))

maf_entity_df.head()

Rows: 1984


,file_id,file_name,case_id,submitter_id,entity_submitter_id,entity_type
0,081c1afc-ea50-4773-9402-dfc686603a16,e35fa380-07ac-45e1-9adc-4351675c6ccf.wxs.aliquot_ensemble_masked.maf.gz,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,TCGA-A8-A09E,TCGA-A8-A09E-10A-01W-A021-09,aliquot
1,081c1afc-ea50-4773-9402-dfc686603a16,e35fa380-07ac-45e1-9adc-4351675c6ccf.wxs.aliquot_ensemble_masked.maf.gz,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,TCGA-A8-A09E,TCGA-A8-A09E-01A-11W-A019-09,aliquot
2,d6709b52-b2fb-4b89-89d4-a426de3f21f0,ac3dbb7f-2ce9-43b7-9f45-1c1521e6bbf5.wxs.aliquot_ensemble_masked.maf.gz,09765b0a-94f6-47d2-af56-93368084ac3a,TCGA-A7-A0CD,TCGA-A7-A0CD-01A-11W-A019-09,aliquot
3,d6709b52-b2fb-4b89-89d4-a426de3f21f0,ac3dbb7f-2ce9-43b7-9f45-1c1521e6bbf5.wxs.aliquot_ensemble_masked.maf.gz,09765b0a-94f6-47d2-af56-93368084ac3a,TCGA-A7-A0CD,TCGA-A7-A0CD-10A-01W-A021-09,aliquot
4,290c5b56-3e55-4bbc-9bcd-2ef0b678490d,a329335e-2962-485d-af32-1f6adf48cf7f.wxs.aliquot_ensemble_masked.maf.gz,0a017f15-1c6b-45e7-8d55-e0a71df1b2e8,TCGA-A8-A08F,TCGA-A8-A08F-01A-11W-A019-09,aliquot


In [44]:
# Identify tumor aliquots associated with each TCGA-BRCA mutation MAF file

maf_entity_df["is_tumor"] = (
    maf_entity_df["entity_submitter_id"]
    .str.split("-")
    .str[3]
    .str.startswith(("01", "02", "03", "04", "05", "06", "07", "08", "09"))
)

maf_entity_df[
    ["submitter_id", "entity_submitter_id", "is_tumor"]
].head(10)

,submitter_id,entity_submitter_id,is_tumor
0,TCGA-A8-A09E,TCGA-A8-A09E-10A-01W-A021-09,False
1,TCGA-A8-A09E,TCGA-A8-A09E-01A-11W-A019-09,True
2,TCGA-A7-A0CD,TCGA-A7-A0CD-01A-11W-A019-09,True
3,TCGA-A7-A0CD,TCGA-A7-A0CD-10A-01W-A021-09,False
4,TCGA-A8-A08F,TCGA-A8-A08F-01A-11W-A019-09,True
5,TCGA-A8-A08F,TCGA-A8-A08F-10A-01W-A021-09,False
6,TCGA-A8-A06T,TCGA-A8-A06T-10A-01W-A021-09,False
7,TCGA-A8-A06T,TCGA-A8-A06T-01A-11W-A019-09,True
8,TCGA-E2-A1L7,TCGA-E2-A1L7-01A-11D-A142-09,True
9,TCGA-E2-A1L7,TCGA-E2-A1L7-10A-01D-A142-09,False


In [45]:
# Keep only tumor aliquots associated with TCGA-BRCA mutation MAF files

tumor_aliquot_df = maf_entity_df[
    maf_entity_df["is_tumor"]
].copy()

print("Tumor aliquot rows:", len(tumor_aliquot_df))

tumor_aliquot_df.head()

Tumor aliquot rows: 992


,file_id,file_name,case_id,submitter_id,entity_submitter_id,entity_type,is_tumor
1,081c1afc-ea50-4773-9402-dfc686603a16,e35fa380-07ac-45e1-9adc-4351675c6ccf.wxs.aliquot_ensemble_masked.maf.gz,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,TCGA-A8-A09E,TCGA-A8-A09E-01A-11W-A019-09,aliquot,True
2,d6709b52-b2fb-4b89-89d4-a426de3f21f0,ac3dbb7f-2ce9-43b7-9f45-1c1521e6bbf5.wxs.aliquot_ensemble_masked.maf.gz,09765b0a-94f6-47d2-af56-93368084ac3a,TCGA-A7-A0CD,TCGA-A7-A0CD-01A-11W-A019-09,aliquot,True
4,290c5b56-3e55-4bbc-9bcd-2ef0b678490d,a329335e-2962-485d-af32-1f6adf48cf7f.wxs.aliquot_ensemble_masked.maf.gz,0a017f15-1c6b-45e7-8d55-e0a71df1b2e8,TCGA-A8-A08F,TCGA-A8-A08F-01A-11W-A019-09,aliquot,True
7,a89b041d-e6cd-4298-93b9-13c62a5f2f01,fe1694df-4aca-4a23-bbf5-c0c1d24d4c2b.wxs.aliquot_ensemble_masked.maf.gz,2c04a2f5-321e-4dea-8e00-268325da65cb,TCGA-A8-A06T,TCGA-A8-A06T-01A-11W-A019-09,aliquot,True
8,7226e23d-6c48-4a24-9c32-f61f3bde7e06,c107d4de-6001-4027-a7f1-7c03d484ddaf.wxs.aliquot_ensemble_masked.maf.gz,791c5768-f0f5-4ab6-86eb-998e5c4b49e3,TCGA-E2-A1L7,TCGA-E2-A1L7-01A-11D-A142-09,aliquot,True


In [46]:
# Check uniqueness of tumor aliquots, mutation files, and TCGA patients

print("Tumor aliquot rows:", len(tumor_aliquot_df))
print("Unique mutation files:", tumor_aliquot_df["file_id"].nunique())
print("Unique tumor aliquots:", tumor_aliquot_df["entity_submitter_id"].nunique())
print("Unique TCGA patients:", tumor_aliquot_df["submitter_id"].nunique())

Tumor aliquot rows: 992
Unique mutation files: 992
Unique tumor aliquots: 992
Unique TCGA patients: 969


In [47]:
# Count how many tumor aliquots are associated with each TCGA-BRCA patient

tumor_aliquots_per_patient = (
    tumor_aliquot_df["submitter_id"]
    .value_counts()
)

tumor_aliquots_per_patient.value_counts().sort_index()

count
1    951
2     13
3      5
Name: count, dtype: int64

In [48]:
# Inspect tumor aliquot identifiers for patients with multiple mutation files

multiple_tumor_aliquots_df = (
    tumor_aliquot_df[
        tumor_aliquot_df["submitter_id"].isin(
            tumor_aliquots_per_patient[
                tumor_aliquots_per_patient > 1
            ].index
        )
    ]
    [
        ["submitter_id", "entity_submitter_id", "file_id", "file_name"]
    ]
    .sort_values(["submitter_id", "entity_submitter_id"])
)

print("Patients with multiple tumor aliquots:",
      multiple_tumor_aliquots_df["submitter_id"].nunique())

print("Tumor aliquots for these patients:",
      len(multiple_tumor_aliquots_df))

multiple_tumor_aliquots_df

Patients with multiple tumor aliquots: 18
Tumor aliquots for these patients: 41


,submitter_id,entity_submitter_id,file_id,file_name
896,TCGA-A7-A0DB,TCGA-A7-A0DB-01A-11D-A272-09,a8976db3-5077-4b8f-9d6c-41a562ae1b4c,d577f725-e910-4a77-89e9-2e982d5493e5.wxs.aliquot_ensemble_masked.maf.gz
1822,TCGA-A7-A0DB,TCGA-A7-A0DB-01A-11W-A019-09,2c9b0f52-0abf-4094-96cb-8511d091ab66,9f77e806-537f-40e9-b33b-b28af74a7a7b.wxs.aliquot_ensemble_masked.maf.gz
1860,TCGA-A7-A0DB,TCGA-A7-A0DB-01C-02D-A272-09,3e7b34fb-a86b-40dc-b349-650301e66e1c,c6810404-b1aa-4669-b91a-d720e93635ec.wxs.aliquot_ensemble_masked.maf.gz
1948,TCGA-A7-A13D,TCGA-A7-A13D-01A-13D-A12Q-09,8f94cc7d-9989-4445-b0b2-90c6d6cb61fa,6a0c8b0a-52d1-4cbb-89cb-06808ee8e7e9.wxs.aliquot_ensemble_masked.maf.gz
1960,TCGA-A7-A13D,TCGA-A7-A13D-01A-13D-A272-09,9a32d563-cd69-4ea5-9b50-c5556d6fda42,764de0ed-ea23-427a-9005-5c5fab299ecd.wxs.aliquot_ensemble_masked.maf.gz
1936,TCGA-A7-A13D,TCGA-A7-A13D-01B-04D-A272-09,1425f01f-b090-48c7-897c-6807939dcba2,62a96ff1-e401-48ee-9f27-b308d672b35c.wxs.aliquot_ensemble_masked.maf.gz
448,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A12Q-09,d8ff1cbb-c551-4585-bdd1-42721e326933,26d5a3d5-2698-4791-8a89-e05ffde7cdca.wxs.aliquot_ensemble_masked.maf.gz
377,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A272-09,26fcd5cb-2178-4884-99e1-630e5f6bf722,09ebc616-2bb3-4779-b7bb-6a4d37cf3d39.wxs.aliquot_ensemble_masked.maf.gz
459,TCGA-A7-A13E,TCGA-A7-A13E-01B-06D-A272-09,9d6db1ea-008d-4f37-b01f-6fb7999a05ba,af7cded7-8cba-42e8-9204-3d50ba53fbe4.wxs.aliquot_ensemble_masked.maf.gz
1486,TCGA-A7-A13G,TCGA-A7-A13G-01A-11D-A13L-09,4c9931f6-8790-497c-8462-9f3ba7bcb9a2,4c6fadb5-27cf-4d3e-9a84-f4744a64f9e5.wxs.aliquot_ensemble_masked.maf.gz


In [49]:
# Display full tumor aliquot identifiers for patients with multiple tumor aliquots

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

multiple_tumor_aliquots_df[
    ["submitter_id", "entity_submitter_id"]
]

,submitter_id,entity_submitter_id
896,TCGA-A7-A0DB,TCGA-A7-A0DB-01A-11D-A272-09
1822,TCGA-A7-A0DB,TCGA-A7-A0DB-01A-11W-A019-09
1860,TCGA-A7-A0DB,TCGA-A7-A0DB-01C-02D-A272-09
1948,TCGA-A7-A13D,TCGA-A7-A13D-01A-13D-A12Q-09
1960,TCGA-A7-A13D,TCGA-A7-A13D-01A-13D-A272-09
1936,TCGA-A7-A13D,TCGA-A7-A13D-01B-04D-A272-09
448,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A12Q-09
377,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A272-09
459,TCGA-A7-A13E,TCGA-A7-A13E-01B-06D-A272-09
1486,TCGA-A7-A13G,TCGA-A7-A13G-01A-11D-A13L-09


In [50]:
# Extract TCGA sample-type codes from the tumor aliquot barcodes

tumor_aliquot_df["sample_code"] = (
    tumor_aliquot_df["entity_submitter_id"]
    .str.split("-")
    .str[3]
    .str[:2]
)

print(tumor_aliquot_df["sample_code"].value_counts().sort_index())

sample_code
01    986
06      6
Name: count, dtype: int64


In [51]:
# Restrict the mutation cohort to primary tumor samples and check patient uniqueness

primary_tumor_df = tumor_aliquot_df[
    tumor_aliquot_df["sample_code"] == "01"
].copy()

print("Primary tumor MAF files:", len(primary_tumor_df))
print("Unique primary tumor aliquots:", primary_tumor_df["entity_submitter_id"].nunique())
print("Unique TCGA patients:", primary_tumor_df["submitter_id"].nunique())

print()
print("Primary tumor files per patient:")
print(
    primary_tumor_df["submitter_id"]
    .value_counts()
    .value_counts()
    .sort_index()
)

Primary tumor MAF files: 986
Unique primary tumor aliquots: 986
Unique TCGA patients: 969

Primary tumor files per patient:
count
1    957
2      7
3      5
Name: count, dtype: int64


In [52]:
# Inspect patients that have multiple primary-tumor mutation MAF files

primary_files_per_patient = (
    primary_tumor_df["submitter_id"]
    .value_counts()
)

multiple_primary_df = (
    primary_tumor_df[
        primary_tumor_df["submitter_id"].isin(
            primary_files_per_patient[
                primary_files_per_patient > 1
            ].index
        )
    ]
    [
        ["submitter_id", "entity_submitter_id", "file_id", "file_name"]
    ]
    .sort_values(["submitter_id", "entity_submitter_id"])
)

print("Patients with multiple primary-tumor files:",
      multiple_primary_df["submitter_id"].nunique())

print("Primary-tumor files for these patients:",
      len(multiple_primary_df))

multiple_primary_df[
    ["submitter_id", "entity_submitter_id"]
]

Patients with multiple primary-tumor files: 12
Primary-tumor files for these patients: 29


,submitter_id,entity_submitter_id
896,TCGA-A7-A0DB,TCGA-A7-A0DB-01A-11D-A272-09
1822,TCGA-A7-A0DB,TCGA-A7-A0DB-01A-11W-A019-09
1860,TCGA-A7-A0DB,TCGA-A7-A0DB-01C-02D-A272-09
1948,TCGA-A7-A13D,TCGA-A7-A13D-01A-13D-A12Q-09
1960,TCGA-A7-A13D,TCGA-A7-A13D-01A-13D-A272-09
1936,TCGA-A7-A13D,TCGA-A7-A13D-01B-04D-A272-09
448,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A12Q-09
377,TCGA-A7-A13E,TCGA-A7-A13E-01A-11D-A272-09
459,TCGA-A7-A13E,TCGA-A7-A13E-01B-06D-A272-09
1486,TCGA-A7-A13G,TCGA-A7-A13G-01A-11D-A13L-09


In [53]:
# Count primary-tumor aliquots for each patient with multiple primary-tumor files

multiple_primary_counts = (
    multiple_primary_df["submitter_id"]
    .value_counts()
    .sort_index()
)

multiple_primary_counts

submitter_id
TCGA-A7-A0DB    3
TCGA-A7-A13D    3
TCGA-A7-A13E    3
TCGA-A7-A13G    2
TCGA-A7-A26E    3
TCGA-A7-A26F    2
TCGA-A7-A26I    2
TCGA-A7-A26J    3
TCGA-AC-A2QH    2
TCGA-AC-A3OD    2
TCGA-AC-A3QQ    2
TCGA-B6-A0I6    2
Name: count, dtype: int64

In [54]:
# Download and inspect one TCGA-BRCA primary-tumor mutation MAF file

test_file_id = primary_tumor_df.iloc[0]["file_id"]
test_file_name = primary_tumor_df.iloc[0]["file_name"]

data_url = f"https://api.gdc.cancer.gov/data/{test_file_id}"

test_maf_response = requests.get(data_url)

print("Status code:", test_maf_response.status_code)
print("File name:", test_file_name)
print("Downloaded bytes:", len(test_maf_response.content))

Status code: 200
File name: e35fa380-07ac-45e1-9adc-4351675c6ccf.wxs.aliquot_ensemble_masked.maf.gz
Downloaded bytes: 18846


In [55]:
# Read the downloaded compressed MAF file into a pandas DataFrame

import io

test_maf_df = pd.read_csv(
    io.BytesIO(test_maf_response.content),
    sep="\t",
    comment="#",
    compression="gzip",
    low_memory=False
)

print("Rows:", len(test_maf_df))
print("Columns:", len(test_maf_df.columns))

test_maf_df.head()

Rows: 51
Columns: 140


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,...,normal_bam_uuid,case_id,GDC_FILTER,COSMIC,hotspot,RNA_Support,RNA_depth,RNA_ref_count,RNA_alt_count,callers
0,PDIK1L,149420,WUGSC,GRCh38,chr1,26122398,26122398,+,Missense_Mutation,SNP,...,ddc0ded6-6188-4977-b0f4-0d3f70239278,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2
1,PIGV,55650,WUGSC,GRCh38,chr1,26797659,26797659,+,Missense_Mutation,SNP,...,ddc0ded6-6188-4977-b0f4-0d3f70239278,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,NaN,COSM1473821,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
2,VCAM1,7412,WUGSC,GRCh38,chr1,100724635,100724635,+,Missense_Mutation,SNP,...,ddc0ded6-6188-4977-b0f4-0d3f70239278,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,NaN,COSM1472331,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
3,BCL9,607,WUGSC,GRCh38,chr1,147620775,147620775,+,Missense_Mutation,SNP,...,ddc0ded6-6188-4977-b0f4-0d3f70239278,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,NaN,COSM1472549,N,Unknown,NaN,NaN,NaN,muse;mutect2;varscan2
4,FLG2,388698,WUGSC,GRCh38,chr1,152351945,152351945,+,Silent,SNP,...,ddc0ded6-6188-4977-b0f4-0d3f70239278,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,NaN,NaN,N,Unknown,NaN,NaN,NaN,muse;mutect2


In [56]:
# Inspect TP53 somatic mutations in the downloaded TCGA-BRCA MAF file

tp53_test = test_maf_df[
    test_maf_df["Hugo_Symbol"] == "TP53"
].copy()

print("TP53 mutation rows:", len(tp53_test))

tp53_test[
    [
        "Hugo_Symbol",
        "Chromosome",
        "Start_Position",
        "Variant_Classification",
        "Variant_Type"
    ]
]

TP53 mutation rows: 0


,Hugo_Symbol,Chromosome,Start_Position,Variant_Classification,Variant_Type


In [57]:
# Identify TP53-relevant mutation annotation columns available in the MAF file

tp53_relevant_columns = [
    col for col in test_maf_df.columns
    if any(
        keyword.lower() in col.lower()
        for keyword in [
            "hgv",
            "protein",
            "transcript",
            "classification",
            "variant_type"
        ]
    )
]

tp53_relevant_columns

['Variant_Classification',
 'Variant_Type',
 'HGVSc',
 'HGVSp',
 'HGVSp_Short',
 'Transcript_ID',
 'Protein_position',
 'TRANSCRIPT_STRAND',
 'TRANSCRIPTION_FACTORS',
 'HGVS_OFFSET']

In [59]:
# Find the first primary-tumor MAF containing TP53 with retry handling

import time

tp53_example_df = None
tp53_example_info = None

for _, row in primary_tumor_df.iterrows():
    file_url = f"https://api.gdc.cancer.gov/data/{row['file_id']}"

    try:
        file_response = requests.get(
            file_url,
            timeout=30
        )

        if file_response.status_code != 200:
            continue

        maf_df = pd.read_csv(
            io.BytesIO(file_response.content),
            sep="\t",
            comment="#",
            compression="gzip",
            low_memory=False
        )

        tp53_rows = maf_df[
            maf_df["Hugo_Symbol"] == "TP53"
        ].copy()

        if len(tp53_rows) > 0:
            tp53_example_df = tp53_rows
            tp53_example_info = row
            break

        time.sleep(0.25)

    except requests.exceptions.RequestException:
        time.sleep(2)
        continue

if tp53_example_df is not None:
    print("Patient:", tp53_example_info["submitter_id"])
    print("Tumor aliquot:", tp53_example_info["entity_submitter_id"])
    print("TP53 mutation rows:", len(tp53_example_df))

    display(
        tp53_example_df[
            [
                "Hugo_Symbol",
                "Chromosome",
                "Start_Position",
                "Variant_Classification",
                "Variant_Type",
                "HGVSc",
                "HGVSp",
                "HGVSp_Short"
            ]
        ]
    )
else:
    print("No TP53-mutated file found in the files successfully checked.")

Patient: TCGA-E2-A1L7
Tumor aliquot: TCGA-E2-A1L7-01A-11D-A142-09
TP53 mutation rows: 1


,Hugo_Symbol,Chromosome,Start_Position,Variant_Classification,Variant_Type,HGVSc,HGVSp,HGVSp_Short
52,TP53,chr17,7674200,Missense_Mutation,SNP,c.763A>T,p.Ile255Phe,p.I255F


In [60]:
# Display the exact TP53 mutation annotation for the identified primary tumor

tp53_example_df[
    [
        "Hugo_Symbol",
        "Chromosome",
        "Start_Position",
        "End_Position",
        "Variant_Classification",
        "Variant_Type",
        "Reference_Allele",
        "Tumor_Seq_Allele2",
        "HGVSc",
        "HGVSp",
        "HGVSp_Short",
        "Protein_position"
    ]
]

,Hugo_Symbol,Chromosome,Start_Position,End_Position,Variant_Classification,Variant_Type,Reference_Allele,Tumor_Seq_Allele2,HGVSc,HGVSp,HGVSp_Short,Protein_position
52,TP53,chr17,7674200,7674200,Missense_Mutation,SNP,T,A,c.763A>T,p.Ile255Phe,p.I255F,255/393


In [61]:
# Extract TP53 somatic mutations from all TCGA-BRCA primary-tumor MAF files

import time

tp53_mutation_rows = []
failed_files = []

total_files = len(primary_tumor_df)

for i, (_, row) in enumerate(primary_tumor_df.iterrows(), start=1):

    file_url = f"https://api.gdc.cancer.gov/data/{row['file_id']}"

    try:
        file_response = requests.get(
            file_url,
            timeout=30
        )

        if file_response.status_code != 200:
            failed_files.append(row["file_id"])
            continue

        maf_df = pd.read_csv(
            io.BytesIO(file_response.content),
            sep="\t",
            comment="#",
            compression="gzip",
            low_memory=False
        )

        tp53_rows = maf_df[
            maf_df["Hugo_Symbol"] == "TP53"
        ].copy()

        if not tp53_rows.empty:
            tp53_rows["file_id"] = row["file_id"]
            tp53_rows["file_name"] = row["file_name"]
            tp53_rows["submitter_id"] = row["submitter_id"]
            tp53_rows["tumor_aliquot_id"] = row["entity_submitter_id"]

            tp53_mutation_rows.append(tp53_rows)

        if i % 100 == 0:
            print(f"Processed {i} of {total_files} files")

        time.sleep(0.25)

    except requests.exceptions.RequestException:
        failed_files.append(row["file_id"])
        time.sleep(2)

print("Files processed:", total_files)
print("Files with connection/download problems:", len(failed_files))
print("Files containing TP53 mutations:", len(tp53_mutation_rows))

Processed 100 of 986 files
Processed 200 of 986 files
Processed 300 of 986 files
Processed 400 of 986 files
Processed 500 of 986 files
Processed 600 of 986 files
Processed 700 of 986 files
Processed 800 of 986 files
Processed 900 of 986 files
Files processed: 986
Files with connection/download problems: 102
Files containing TP53 mutations: 304


In [62]:
# Retry TCGA-BRCA mutation files that failed during the first TP53 extraction pass

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

session = requests.Session()
session.mount(
    "https://",
    HTTPAdapter(max_retries=retry_strategy)
)

retry_failed_files = []

failed_primary_df = primary_tumor_df[
    primary_tumor_df["file_id"].isin(failed_files)
].copy()

print("Files to retry:", len(failed_primary_df))

for i, (_, row) in enumerate(failed_primary_df.iterrows(), start=1):

    file_url = f"https://api.gdc.cancer.gov/data/{row['file_id']}"

    try:
        file_response = session.get(
            file_url,
            timeout=60
        )

        if file_response.status_code != 200:
            retry_failed_files.append(row["file_id"])
            continue

        maf_df = pd.read_csv(
            io.BytesIO(file_response.content),
            sep="\t",
            comment="#",
            compression="gzip",
            low_memory=False
        )

        tp53_rows = maf_df[
            maf_df["Hugo_Symbol"] == "TP53"
        ].copy()

        if not tp53_rows.empty:
            tp53_rows["file_id"] = row["file_id"]
            tp53_rows["file_name"] = row["file_name"]
            tp53_rows["submitter_id"] = row["submitter_id"]
            tp53_rows["tumor_aliquot_id"] = row["entity_submitter_id"]

            tp53_mutation_rows.append(tp53_rows)

        if i % 20 == 0:
            print(f"Retried {i} of {len(failed_primary_df)} files")

        time.sleep(0.5)

    except requests.exceptions.RequestException:
        retry_failed_files.append(row["file_id"])

print()
print("Retried files:", len(failed_primary_df))
print("Files still failing:", len(retry_failed_files))
print("Files containing TP53 mutations so far:", len(tp53_mutation_rows))

Files to retry: 102
Retried 20 of 102 files
Retried 40 of 102 files
Retried 60 of 102 files
Retried 80 of 102 files
Retried 100 of 102 files

Retried files: 102
Files still failing: 0
Files containing TP53 mutations so far: 340


In [63]:
# Combine all TP53 mutation records and summarize TP53-altered patients

tp53_mutations_df = pd.concat(
    tp53_mutation_rows,
    ignore_index=True
)

print("Total TP53 mutation rows:", len(tp53_mutations_df))
print("TP53-mutated MAF files:", tp53_mutations_df["file_id"].nunique())
print("Unique TP53-mutated patients:", tp53_mutations_df["submitter_id"].nunique())

tp53_mutations_df[
    [
        "submitter_id",
        "tumor_aliquot_id",
        "Hugo_Symbol",
        "Variant_Classification",
        "HGVSp_Short"
    ]
].head()

Total TP53 mutation rows: 347
TP53-mutated MAF files: 340
Unique TP53-mutated patients: 332


,submitter_id,tumor_aliquot_id,Hugo_Symbol,Variant_Classification,HGVSp_Short
0,TCGA-E2-A1L7,TCGA-E2-A1L7-01A-11D-A142-09,TP53,Missense_Mutation,p.I255F
1,TCGA-PL-A8LV,TCGA-PL-A8LV-01A-21D-A41F-09,TP53,Missense_Mutation,p.C275Y
2,TCGA-A2-A0SV,TCGA-A2-A0SV-01A-11D-A099-09,TP53,In_Frame_Del,p.F113_H115del
3,TCGA-E9-A6HE,TCGA-E9-A6HE-01A-11D-A31U-09,TP53,Nonsense_Mutation,p.R306*
4,TCGA-AO-A124,TCGA-AO-A124-01A-11D-A10M-09,TP53,Nonsense_Mutation,p.R213*


In [64]:
# Print summary counts for TP53 mutations and TP53-altered patients

print("Total TP53 mutation rows:", len(tp53_mutations_df))
print("TP53-mutated MAF files:", tp53_mutations_df["file_id"].nunique())
print("Unique TP53-mutated patients:", tp53_mutations_df["submitter_id"].nunique())

Total TP53 mutation rows: 347
TP53-mutated MAF files: 340
Unique TP53-mutated patients: 332


In [65]:
# Count the types of TP53 somatic mutations detected in TCGA-BRCA

tp53_mutations_df["Variant_Classification"].value_counts()

Variant_Classification
Missense_Mutation    203
Nonsense_Mutation     49
Frame_Shift_Del       49
Splice_Site           24
Frame_Shift_Ins       15
In_Frame_Del           6
Splice_Region          1
Name: count, dtype: int64

In [66]:
# Create patient-level TP53 mutation status for mutation-assessed TCGA-BRCA patients

mutation_assessed_patients = (
    primary_tumor_df[
        ["case_id", "submitter_id"]
    ]
    .drop_duplicates()
    .copy()
)

tp53_altered_patients = set(
    tp53_mutations_df["submitter_id"].unique()
)

mutation_assessed_patients["tp53_status"] = (
    mutation_assessed_patients["submitter_id"]
    .apply(
        lambda patient:
        "TP53-altered"
        if patient in tp53_altered_patients
        else "TP53-WT"
    )
)

print("Mutation-assessed patients:", len(mutation_assessed_patients))
print()
print(mutation_assessed_patients["tp53_status"].value_counts())

Mutation-assessed patients: 969

tp53_status
TP53-WT         637
TP53-altered    332
Name: count, dtype: int64


In [67]:
# Merge patient-level TP53 mutation status into the TCGA-BRCA clinical cohort

clinical_tp53_df = clinical_df.merge(
    mutation_assessed_patients[
        ["case_id", "submitter_id", "tp53_status"]
    ],
    on=["case_id", "submitter_id"],
    how="left"
)

clinical_tp53_df["tp53_status"] = (
    clinical_tp53_df["tp53_status"]
    .fillna("Not assessed")
)

print("Total clinical patients:", len(clinical_tp53_df))
print()
print(clinical_tp53_df["tp53_status"].value_counts())

Total clinical patients: 1098

tp53_status
TP53-WT         637
TP53-altered    332
Not assessed    129
Name: count, dtype: int64


In [68]:
# Calculate TP53 alteration frequency among mutation-assessed TCGA-BRCA patients

assessed_tp53_df = clinical_tp53_df[
    clinical_tp53_df["tp53_status"] != "Not assessed"
].copy()

tp53_frequency = (
    assessed_tp53_df["tp53_status"]
    .value_counts()
    .to_frame("patient_count")
)

tp53_frequency["percentage"] = (
    tp53_frequency["patient_count"]
    / len(assessed_tp53_df)
    * 100
).round(1)

print("Mutation-assessed patients:", len(assessed_tp53_df))

tp53_frequency

Mutation-assessed patients: 969


,patient_count,percentage
tp53_status,,
TP53-WT,637,65.7
TP53-altered,332,34.3


In [69]:
# Save the TCGA-BRCA clinical cohort with patient-level TP53 mutation status

output_path = "../data/processed/tcga_brca_clinical_tp53_status.csv"

clinical_tp53_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(clinical_tp53_df))
print("Columns:", len(clinical_tp53_df.columns))

Saved: ../data/processed/tcga_brca_clinical_tp53_status.csv
Rows: 1098
Columns: 8


In [70]:
# Save the detailed TP53 somatic mutation records for downstream analysis

tp53_mutation_output_path = "../data/processed/tcga_brca_tp53_mutations.csv"

tp53_mutations_df.to_csv(
    tp53_mutation_output_path,
    index=False
)

print("Saved:", tp53_mutation_output_path)
print("Rows:", len(tp53_mutations_df))
print("Unique patients:", tp53_mutations_df["submitter_id"].nunique())

Saved: ../data/processed/tcga_brca_tp53_mutations.csv
Rows: 347
Unique patients: 332


In [71]:
# Verify the saved TCGA-BRCA TP53 cohort and mutation files

saved_clinical_tp53_df = pd.read_csv(
    "../data/processed/tcga_brca_clinical_tp53_status.csv"
)

saved_tp53_mutations_df = pd.read_csv(
    "../data/processed/tcga_brca_tp53_mutations.csv"
)

print("Clinical TP53 table:")
print("Rows:", len(saved_clinical_tp53_df))
print(saved_clinical_tp53_df["tp53_status"].value_counts())

print()

print("TP53 mutation table:")
print("Rows:", len(saved_tp53_mutations_df))
print("Unique patients:", saved_tp53_mutations_df["submitter_id"].nunique())

Clinical TP53 table:
Rows: 1098
tp53_status
TP53-WT         637
TP53-altered    332
Not assessed    129
Name: count, dtype: int64

TP53 mutation table:
Rows: 347
Unique patients: 332
